In [2]:
import pandas as pd

# -----------------------------
# Configuration
# -----------------------------
PAYSIM_PATH = "PS_20174392719_1491204439457_log.csv"   # Change if needed
NUM_CUSTOMERS = 8000
OUTPUT_FILE = "bank_transactions.csv"
RANDOM_STATE = 42

# -----------------------------
# Read PaySim
# -----------------------------
print("Reading PaySim dataset...")

df = pd.read_csv('paysim dataset.csv')

print(f"Total Transactions : {len(df):,}")

# -----------------------------
# Find all customer accounts
# -----------------------------
customers = df["nameOrig"].unique()

print(f"Unique Customers : {len(customers):,}")

# -----------------------------
# Randomly choose customers
# -----------------------------
selected_customers = (
    pd.Series(customers)
    .sample(NUM_CUSTOMERS, random_state=RANDOM_STATE)
)

print(f"Selected Customers : {len(selected_customers):,}")

# -----------------------------
# Keep all transactions
# -----------------------------
bank_df = df[df["nameOrig"].isin(selected_customers)].copy()

print(f"Transactions After Filtering : {len(bank_df):,}")

# -----------------------------
# Save
# -----------------------------
bank_df.to_csv(OUTPUT_FILE, index=False)

print("\nDone.")
print(f"Saved to {OUTPUT_FILE}")

Reading PaySim dataset...
Total Transactions : 6,362,620
Unique Customers : 6,353,307
Selected Customers : 8,000
Transactions After Filtering : 8,009

Done.
Saved to bank_transactions.csv


# Generating our new Dataset:

In [3]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

In [6]:
CHUNK_SIZE = 100000

sampled_chunks = []

columns = [
    "step",
    "type",
    "amount",
    "nameOrig",
    "oldbalanceOrg",
    "newbalanceOrig",
    "nameDest",
    "oldbalanceDest",
    "newbalanceDest",
    "isFraud"
]

for chunk in pd.read_csv(
    "paysim dataset.csv",
    usecols=columns,
    chunksize=CHUNK_SIZE
):
    sampled = chunk.sample(
        frac=0.02,
        random_state=42
    )

    sampled_chunks.append(sampled)

bank_df = pd.concat(
    sampled_chunks,
    ignore_index=True
)

print(bank_df.shape)

(127252, 10)


In [7]:
bank_df = bank_df.sample(
    n=100000,
    random_state=42
).reset_index(drop=True)

print(bank_df.shape)

(100000, 10)


In [8]:
bank_df.to_csv(
    "bank_transactions_raw.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


## Refining the new csv:

In [9]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

bank_df = pd.read_csv("bank_transactions_raw.csv")

print(bank_df.shape)

bank_df.head()


(100000, 10)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,254,PAYMENT,3853.04,C1469510588,0.0,0.00,M767993324,0.00,0.00,0
1,302,CASH_IN,278263.77,C1394025052,204530.0,482793.77,C389961354,300561.56,22297.78,0
2,39,CASH_OUT,328073.84,C2032213545,15865.0,0.00,C1021427324,2361146.10,2689219.94,0
3,253,TRANSFER,85912.64,C1138075457,0.0,0.00,C150003255,1555431.93,1641344.57,0
4,185,PAYMENT,17991.22,C1158304344,0.0,0.00,M1365982848,0.00,0.00,0


In [10]:
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)

bank_df = pd.read_csv("bank_transactions_raw.csv")

print(bank_df.shape)

bank_df.head()

(100000, 10)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,254,PAYMENT,3853.04,C1469510588,0.0,0.00,M767993324,0.00,0.00,0
1,302,CASH_IN,278263.77,C1394025052,204530.0,482793.77,C389961354,300561.56,22297.78,0
2,39,CASH_OUT,328073.84,C2032213545,15865.0,0.00,C1021427324,2361146.10,2689219.94,0
3,253,TRANSFER,85912.64,C1138075457,0.0,0.00,C150003255,1555431.93,1641344.57,0
4,185,PAYMENT,17991.22,C1158304344,0.0,0.00,M1365982848,0.00,0.00,0


In [15]:
print(bank_df.columns.tolist())

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud']


In [16]:
sender_ids = set(bank_df["nameOrig"].unique())

receiver_ids = set(bank_df["nameDest"].unique())

all_customers = list(sender_ids.union(receiver_ids))

print("Unique Sender IDs :", len(sender_ids))
print("Unique Receiver IDs :", len(receiver_ids))
print("Total Unique Customers :", len(all_customers))

Unique Sender IDs : 99998
Unique Receiver IDs : 92816
Total Unique Customers : 192810


In [17]:
customer_master = pd.DataFrame({
    "Paysim_ID": all_customers
})

customer_master.head()

,Paysim_ID
0,C980470768
1,C1661072942
2,C1278584897
3,C1504785763
4,C104905172


In [18]:
customer_master["Customer_ID"] = [
    f"CUST{i:07d}"
    for i in range(1, len(customer_master) + 1)
]

customer_master.head()

,Paysim_ID,Customer_ID
0,C980470768,CUST0000001
1,C1661072942,CUST0000002
2,C1278584897,CUST0000003
3,C1504785763,CUST0000004
4,C104905172,CUST0000005


In [19]:
import string

In [20]:
banks = [
    "State Bank of India",
    "HDFC Bank",
    "ICICI Bank",
    "Axis Bank",
    "Punjab National Bank",
    "Bank of Baroda",
    "Canara Bank",
    "Union Bank of India",
    "Kotak Mahindra Bank",
    "IndusInd Bank"
]

In [21]:
ifsc_codes = {
    "State Bank of India":"SBIN",
    "HDFC Bank":"HDFC",
    "ICICI Bank":"ICIC",
    "Axis Bank":"UTIB",
    "Punjab National Bank":"PUNB",
    "Bank of Baroda":"BARB",
    "Canara Bank":"CNRB",
    "Union Bank of India":"UBIN",
    "Kotak Mahindra Bank":"KKBK",
    "IndusInd Bank":"INDB"
}

In [22]:
cities = [
    "Surat",
    "Ahmedabad",
    "Mumbai",
    "Delhi",
    "Pune",
    "Jaipur",
    "Lucknow",
    "Indore",
    "Bhopal",
    "Nagpur",
    "Hyderabad",
    "Bengaluru",
    "Chennai",
    "Kolkata",
    "Patna"
]

In [23]:
customer_master["Bank_Name"] = np.random.choice(
    banks,
    len(customer_master)
)

customer_master.head()

,Paysim_ID,Customer_ID,Bank_Name
0,C980470768,CUST0000001,Canara Bank
1,C1661072942,CUST0000002,Axis Bank
2,C1278584897,CUST0000003,Union Bank of India
3,C1504785763,CUST0000004,Punjab National Bank
4,C104905172,CUST0000005,Canara Bank


In [24]:
customer_master["City"] = np.random.choice(
    cities,
    len(customer_master)
)

customer_master.head()

,Paysim_ID,Customer_ID,Bank_Name,City
0,C980470768,CUST0000001,Canara Bank,Bhopal
1,C1661072942,CUST0000002,Axis Bank,Pune
2,C1278584897,CUST0000003,Union Bank of India,Chennai
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai
4,C104905172,CUST0000005,Canara Bank,Kolkata


In [25]:
customer_master["Account_Number"] = [
    str(random.randint(10**11,10**12-1))
    for _ in range(len(customer_master))
]

In [26]:
customer_master["IFSC"] = customer_master["Bank_Name"].apply(
    lambda x:
    ifsc_codes[x] + str(random.randint(1000000,9999999))
)

In [27]:
customer_master["Phone_Number"] = [
    str(random.randint(6000000000,9999999999))
    for _ in range(len(customer_master))
]

In [28]:
providers = [
    "oksbi",
    "okhdfcbank",
    "okicici",
    "ybl",
    "ibl"
]

customer_master["UPI_ID"] = customer_master.apply(
    lambda row:
    row["Phone_Number"] +
    "@" +
    random.choice(providers),
    axis=1
)

In [29]:
customer_master.head()

,Paysim_ID,Customer_ID,Bank_Name,City,Account_Number,IFSC,Phone_Number,UPI_ID
0,C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank
1,C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi
2,C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl
4,C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl


In [31]:
male_names = [
    "Aarav", "Aaryan", "Aashish", "Abhay", "Abhinav", "Abhishek", "Aditya",
    "Advait", "Ajay", "Akash", "Akhil", "Akshay", "Amit", "Amol", "Anand",
    "Anil", "Ankit", "Ansh", "Anupam", "Anurag", "Aravind", "Arjun", "Arnav",
    "Arun", "Arvind", "Aryan", "Ashok", "Ashwin", "Atharva", "Avinash", "Ayush",
    "Badrinath", "Balaji", "Bharat", "Bhaskar", "Bhuvan", "Bipin", "Brahma",
    "Chaitanya", "Chandan", "Chandra", "Charan", "Chetan", "Chirag", "Chitta",
    "Daksh", "Darshan", "Deepak", "Dev", "Devansh", "Deven", "Dhruv", "Dilip",
    "Dinesh", "Dipak", "Divyansh", "Eeshwar", "Ekansh", "Eklavya", "Farhan",
    "Feroz", "Gagan", "Ganesh", "Gaurav", "Gautam", "Girish", "Gopal",
    "Govind", "Gulshan", "Hardik", "Hari", "Harish", "Harsh", "Harshad",
    "Harshil", "Hemant", "Himesh", "Hitesh", "Hrithik", "Ishaan", "Ishwar",
    "Imran", "Inder", "Jagadish", "Jai", "Jaideep", "Jatin", "Jay", "Jayant",
    "Jeevan", "Jignesh", "Jitesh", "Jyotir", "Kabir", "Kailash", "Kalyan",
    "Kamal", "Karan", "Karthik", "Kartik", "Kaushal", "Kaushik", "Kavin",
    "Kedar", "Keshav", "Ketan", "Kiran", "Kishore", "Krishna", "Krunal",
    "Kumar", "Kunal", "Kush", "Lakshya", "Lalit", "Lokesh", "Luv", "Madhav",
    "Mahesh", "Manav", "Manish", "Manoj", "Mayank", "Meet", "Mehul", "Mihir",
    "Milan", "Milind", "Mitul", "Mohan", "Mohit", "Mukesh", "Nachiket",
    "Nakul", "Naman", "Narayan", "Narendra", "Naresh", "Naveen", "Navin",
    "Neeraj", "Nikhil", "Nikunj", "Nilesh", "Niraj", "Nirav", "Nirmal",
    "Nishant", "Nitin", "Om", "Omkar", "Ojas", "Pankaj", "Parag", "Param",
    "Parth", "Pavan", "Piyush", "Prabhat", "Prabhu", "Pradeep", "Pranav",
    "Pranay", "Pratik", "Praveen", "Prem", "Pritam", "Puneet", "Pushkar",
    "Rachit", "Raghav", "Rahul", "Raj", "Rajan", "Rajat", "Rajeev", "Rajesh",
    "Rakesh", "Ram", "Raman", "Ramesh", "Randhir", "Ravi", "Ravindra",
    "Reyansh", "Rishi", "Ritik", "Rohan", "Rohit", "Roshan", "Rupesh",
    "Sachin", "Sagar", "Sahil", "Samir", "Sanjay", "Sanjeev", "Sanket",
    "Sarthak", "Saurabh"
]

In [32]:
female_names = [
    "Aadhya", "Aalia", "Aanya", "Aaradhya", "Aarthi", "Aashi", "Aayushi", "Aditi", "Advika", "Aishwarya",
    "Akanksha", "Akhila", "Akshara", "Alia", "Amira", "Amrita", "Ananya", "Anika", "Anita", "Anjali",
    "Ankita", "Anshika", "Anushka", "Aparna", "Aradhana", "Arpita", "Asha", "Ashwini", "Avani", "Ayushi",
    "Bhavana", "Bhavani", "Bhavya", "Bhoomi", "Bipasha", "Chaitali", "Chandani", "Charu", "Chetana", "Chhavi",
    "Daksha", "Damini", "Darshana", "Deepa", "Deepali", "Deepika", "Deepti", "Devanshi", "Devika", "Dhanya",
    "Dhara", "Drishti", "Esha", "Ekta", "Falguni", "Farah", "Gargi", "Garima", "Gauri", "Gayatri",
    "Geeta", "Geetanjali", "Gunjan", "Harshita", "Hema", "Hemalatha", "Himani", "Indira", "Indu", "Isha",
    "Ishani", "Ishika", "Ishita", "Jagruti", "Jahnavi", "Janki", "Jaya", "Jayashree", "Jia", "Juhi",
    "Jyoti", "Jyotsna", "Kajal", "Kalpana", "Kalyani", "Kamala", "Kanika", "Kareena", "Karishma", "Kavita",
    "Kavya", "Khushi", "Kiran", "Kirti", "Komal", "Kriti", "Krutika", "Kusum", "Lata", "Latika",
    "Laxmi", "Leela", "Madhuri", "Mahima", "Maithili", "Malavika", "Malini", "Mamata", "Manasi", "Manisha",
    "Manju", "Meena", "Meenakshi", "Meera", "Megha", "Meghna", "Mitali", "Mohini", "Monica", "Mrinali",
    "Nalini", "Namita", "Namrata", "Nandini", "Nandita", "Navya", "Neelam", "Neha", "Nidhi", "Niharika",
    "Nikita", "Nilima", "Nisha", "Nita", "Nivedita", "Niyati", "Nupur", "Nutan", "Ojaswini", "Padma",
    "Pallavi", "Pari", "Parvati", "Payal", "Pooja", "Poonam", "Poornima", "Prachi", "Pragya", "Prerana",
    "Priti", "Priya", "Priyanka", "Rachana", "Radha", "Radhika", "Ragini", "Rajani", "Rakhi", "Ramya",
    "Rani", "Rashmi", "Raveena", "Rekha", "Renu", "Renuka", "Reshma", "Richa", "Riddhi", "Riya",
    "Rohini", "Roshni", "Ruchi", "Rupali", "Saanvi", "Sachi", "Sadhana", "Sakshi", "Saloni", "Sangeeta",
    "Sanjana", "Sapna", "Saraswati", "Sarika", "Sarita", "Seema", "Shalini", "Shikha", "Shilpa", "Shreya",
    "Shruti", "Smriti", "Sneha", "Sonali", "Sonia", "Srishti", "Suman", "Sunita", "Sushma", "Swati"
]

In [33]:
last_names = [
"Sharma","Verma","Gupta","Patel","Shah","Bhatt","Mehta","Desai","Joshi","Pandya",
"Trivedi","Dave","Parmar","Chauhan","Solanki","Rathod","Yadav","Singh","Kaur","Gill",
"Sandhu","Kapoor","Malhotra","Arora","Bansal","Agarwal","Jain","Mittal","Saxena","Tiwari",
"Mishra","Pandey","Dubey","Tripathi","Chaturvedi","Srivastava","Roy","Das","Bose","Ghosh",
"Banerjee","Chatterjee","Mukherjee","Dutta","Reddy","Naidu","Rao","Nair","Menon","Iyer",
"Krishnan","Pillai","Subramanian","Kulkarni","Patil","Deshmukh","Pawar","Sawant","More","Shinde",
"Naik","Shetty","Hegde","Acharya","Bora","Kalita","Ali","Khan","Ansari","Qureshi",
"Shaikh","Pathan","Syed","Thomas","Mathew","Varghese","Fernandes","D'Souza","Pereira","Lal",
"Rawat","Negi","Bisht","Mahajan","Soni","Chawla","Kohli","Bhatia","Goel","Bagchi",
"Kar","Sen","Paul","Prasad","Sahu","Behera","Mohanty","Barik","Majumdar","Bhattacharya"
]

In [34]:
customer_master["Gender"] = np.random.choice(
    ["Male", "Female"],
    len(customer_master),
    p=[0.52, 0.48]
)

In [35]:
def generate_name(gender):

    if gender == "Male":
        first = random.choice(male_names)
    else:
        first = random.choice(female_names)

    last = random.choice(last_names)

    return f"{first} {last}"

customer_master["Customer_Name"] = customer_master["Gender"].apply(generate_name)

In [36]:
customer_master[["Customer_Name","Gender"]].sample(20)

,Customer_Name,Gender
66210,Divyansh Kalita,Male
45484,Rakhi Iyer,Female
182806,Riya Das,Female
36812,Renu Bisht,Female
173100,Sarika Kapoor,Female
173447,Anupam Sawant,Male
11666,Kumar Qureshi,Male
119444,Ishika Tiwari,Female
9656,Devika Tiwari,Female
151752,Kabir Bose,Male


In [37]:
email_domains = [
    "gmail.com",
    "yahoo.com",
    "outlook.com",
    "hotmail.com",
    "icloud.com"
]

customer_master["Email"] = customer_master.apply(
    lambda row:
    row["Customer_Name"].lower().replace(" ", ".")
    + str(random.randint(1,999))
    + "@"
    + random.choice(email_domains),
    axis=1
)

In [38]:
from datetime import datetime, timedelta

today = datetime.today()

dob = []

for _ in range(len(customer_master)):

    age = random.randint(18,75)

    days = random.randint(0,365)

    birth = today - timedelta(days=age*365+days)

    dob.append(birth.date())

customer_master["DOB"] = dob

In [39]:
joined = []

for _ in range(len(customer_master)):

    years = random.randint(1,15)

    days = random.randint(0,365)

    d = today - timedelta(days=years*365+days)

    joined.append(d.date())

customer_master["Customer_Since"] = joined

In [40]:
occupations = [

"Student",
"Engineer",
"Doctor",
"Teacher",
"Business",
"Lawyer",
"Farmer",
"Government Employee",
"Private Employee",
"Self Employed",
"Accountant",
"Sales Executive",
"Software Developer",
"Consultant",
"Retired",
"Police Officer",
"Professor",
"Shop Owner",
"Driver",
"Unemployed"

]

customer_master["Occupation"] = np.random.choice(
    occupations,
    len(customer_master)
)

In [41]:
customer_master["Annual_Income"] = np.random.randint(
    150000,
    4000000,
    len(customer_master)
)

In [42]:
customer_master["Annual_Income"] = np.random.randint(
    150000,
    4000000,
    len(customer_master)
)

In [43]:
customer_master["KYC_Status"] = np.random.choice(

["Verified","Pending"],

len(customer_master),

p=[0.98,0.02]

)

In [44]:
customer_master["Aadhaar"] = [

str(random.randint(100000000000,999999999999))

for _ in range(len(customer_master))

]

In [45]:
letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

customer_master["PAN"] = [

random.choice(letters)
+ random.choice(letters)
+ random.choice(letters)
+ random.choice(letters)
+ random.choice(letters)
+ str(random.randint(1000,9999))
+ random.choice(letters)

for _ in range(len(customer_master))

]

In [46]:
customer_master["Device_ID"] = [

"DEV"+str(random.randint(10000000,99999999))

for _ in range(len(customer_master))

]

In [47]:
customer_master["IMEI"] = [

str(random.randint(100000000000000,999999999999999))

for _ in range(len(customer_master))

]

In [48]:
customer_master["SIM_Number"] = [

str(random.randint(8900000000000000000,8999999999999999999))

for _ in range(len(customer_master))

]

In [52]:
customer_master.to_csv(

"customer_master.csv",

index=False

)

print(customer_master.shape)

customer_master.head()

(192810, 21)


,Paysim_ID,Customer_ID,Bank_Name,City,Account_Number,IFSC,Phone_Number,UPI_ID,Gender,Customer_Name,...,DOB,Customer_Since,Occupation,Annual_Income,KYC_Status,Aadhaar,PAN,Device_ID,IMEI,SIM_Number
0,C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank,Male,Niraj Lal,...,1979-01-15,2017-10-08,Engineer,3733963,Verified,592337894135,GYLNC8130S,DEV46180493,704849945366304,8908844093645845822
1,C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi,Male,Raghav Kaur,...,1962-11-01,2023-04-30,Business,1024034,Verified,804901053843,TJUFF8228U,DEV18471166,476901535879387,8971533799543255778
2,C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank,Male,Eklavya Srivastava,...,1993-01-12,2011-09-27,Student,2396859,Verified,846416088312,TJAJK8210S,DEV88761645,313922826100062,8975770667602947445
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl,Female,Sunita Roy,...,1998-11-08,2010-09-11,Student,2355699,Verified,738059893920,CAUZP7040Z,DEV93987154,419365274232192,8906655297104633890
4,C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl,Male,Aashish Banerjee,...,1981-09-04,2024-11-18,Accountant,476100,Verified,276877127316,SENYG7140Z,DEV90720393,669835457460092,8993517761120241202


In [53]:
customer_master.to_csv(

"customer_master.csv",

index=False

)

print(customer_master.shape)

customer_master.head()

(192810, 21)


,Paysim_ID,Customer_ID,Bank_Name,City,Account_Number,IFSC,Phone_Number,UPI_ID,Gender,Customer_Name,...,DOB,Customer_Since,Occupation,Annual_Income,KYC_Status,Aadhaar,PAN,Device_ID,IMEI,SIM_Number
0,C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank,Male,Niraj Lal,...,1979-01-15,2017-10-08,Engineer,3733963,Verified,592337894135,GYLNC8130S,DEV46180493,704849945366304,8908844093645845822
1,C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi,Male,Raghav Kaur,...,1962-11-01,2023-04-30,Business,1024034,Verified,804901053843,TJUFF8228U,DEV18471166,476901535879387,8971533799543255778
2,C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank,Male,Eklavya Srivastava,...,1993-01-12,2011-09-27,Student,2396859,Verified,846416088312,TJAJK8210S,DEV88761645,313922826100062,8975770667602947445
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl,Female,Sunita Roy,...,1998-11-08,2010-09-11,Student,2355699,Verified,738059893920,CAUZP7040Z,DEV93987154,419365274232192,8906655297104633890
4,C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl,Male,Aashish Banerjee,...,1981-09-04,2024-11-18,Accountant,476100,Verified,276877127316,SENYG7140Z,DEV90720393,669835457460092,8993517761120241202


In [55]:
customer_master = pd.read_csv("customer_master.csv")

In [56]:
customer_lookup = customer_master.set_index("Paysim_ID")

In [57]:
print(customer_lookup.shape)

customer_lookup.head()

(192810, 20)


,Customer_ID,Bank_Name,City,Account_Number,IFSC,Phone_Number,UPI_ID,Gender,Customer_Name,Email,DOB,Customer_Since,Occupation,Annual_Income,KYC_Status,Aadhaar,PAN,Device_ID,IMEI,SIM_Number
Paysim_ID,,,,,,,,,,,,,,,,,,,,
C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank,Male,Niraj Lal,niraj.lal636@outlook.com,1979-01-15,2017-10-08,Engineer,3733963,Verified,592337894135,GYLNC8130S,DEV46180493,704849945366304,8908844093645845822
C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi,Male,Raghav Kaur,raghav.kaur589@outlook.com,1962-11-01,2023-04-30,Business,1024034,Verified,804901053843,TJUFF8228U,DEV18471166,476901535879387,8971533799543255778
C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank,Male,Eklavya Srivastava,eklavya.srivastava344@icloud.com,1993-01-12,2011-09-27,Student,2396859,Verified,846416088312,TJAJK8210S,DEV88761645,313922826100062,8975770667602947445
C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl,Female,Sunita Roy,sunita.roy23@yahoo.com,1998-11-08,2010-09-11,Student,2355699,Verified,738059893920,CAUZP7040Z,DEV93987154,419365274232192,8906655297104633890
C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl,Male,Aashish Banerjee,aashish.banerjee847@hotmail.com,1981-09-04,2024-11-18,Accountant,476100,Verified,276877127316,SENYG7140Z,DEV90720393,669835457460092,8993517761120241202


In [58]:
customer_master.columns

Index(['Paysim_ID', 'Customer_ID', 'Bank_Name', 'City', 'Account_Number',
       'IFSC', 'Phone_Number', 'UPI_ID', 'Gender', 'Customer_Name', 'Email',
       'DOB', 'Customer_Since', 'Occupation', 'Annual_Income', 'KYC_Status',
       'Aadhaar', 'PAN', 'Device_ID', 'IMEI', 'SIM_Number'],
      dtype='object')

In [59]:
bank_transactions = bank_df.copy()

In [60]:
sender_master = customer_master.copy()

sender_master = sender_master.rename(
    columns={
        col: "Sender_" + col
        for col in sender_master.columns
        if col != "Paysim_ID"
    }
)

sender_master.head()

,Paysim_ID,Sender_Customer_ID,Sender_Bank_Name,Sender_City,Sender_Account_Number,Sender_IFSC,Sender_Phone_Number,Sender_UPI_ID,Sender_Gender,Sender_Customer_Name,...,Sender_DOB,Sender_Customer_Since,Sender_Occupation,Sender_Annual_Income,Sender_KYC_Status,Sender_Aadhaar,Sender_PAN,Sender_Device_ID,Sender_IMEI,Sender_SIM_Number
0,C980470768,CUST0000001,Canara Bank,Bhopal,223005401501,CNRB4034684,9876137445,9876137445@okhdfcbank,Male,Niraj Lal,...,1979-01-15,2017-10-08,Engineer,3733963,Verified,592337894135,GYLNC8130S,DEV46180493,704849945366304,8908844093645845822
1,C1661072942,CUST0000002,Axis Bank,Pune,911856239313,UTIB2847846,7675203574,7675203574@oksbi,Male,Raghav Kaur,...,1962-11-01,2023-04-30,Business,1024034,Verified,804901053843,TJUFF8228U,DEV18471166,476901535879387,8971533799543255778
2,C1278584897,CUST0000003,Union Bank of India,Chennai,367469214295,UBIN7124901,9802299081,9802299081@okhdfcbank,Male,Eklavya Srivastava,...,1993-01-12,2011-09-27,Student,2396859,Verified,846416088312,TJAJK8210S,DEV88761645,313922826100062,8975770667602947445
3,C1504785763,CUST0000004,Punjab National Bank,Mumbai,251282538206,PUNB6347579,8307012433,8307012433@ibl,Female,Sunita Roy,...,1998-11-08,2010-09-11,Student,2355699,Verified,738059893920,CAUZP7040Z,DEV93987154,419365274232192,8906655297104633890
4,C104905172,CUST0000005,Canara Bank,Kolkata,214832269481,CNRB2946693,6539439394,6539439394@ibl,Male,Aashish Banerjee,...,1981-09-04,2024-11-18,Accountant,476100,Verified,276877127316,SENYG7140Z,DEV90720393,669835457460092,8993517761120241202


In [61]:
bank_transactions = bank_transactions.merge(
    sender_master,
    left_on="nameOrig",
    right_on="Paysim_ID",
    how="left"
)

In [62]:
bank_transactions.drop(
    columns=["Paysim_ID"],
    inplace=True
)

In [63]:
receiver_master = customer_master.copy()

receiver_master = receiver_master.rename(
    columns={
        col: "Receiver_" + col
        for col in receiver_master.columns
        if col != "Paysim_ID"
    }
)

In [64]:
bank_transactions = bank_transactions.merge(
    receiver_master,
    left_on="nameDest",
    right_on="Paysim_ID",
    how="left"
)

In [65]:
bank_transactions.drop(
    columns=["Paysim_ID"],
    inplace=True
)

In [66]:
bank_transactions.drop(
    columns=[
        "nameOrig",
        "nameDest"
    ],
    inplace=True
)

In [67]:
print(bank_transactions.shape)

bank_transactions.head()

(100000, 48)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,Sender_Customer_ID,Sender_Bank_Name,...,Receiver_DOB,Receiver_Customer_Since,Receiver_Occupation,Receiver_Annual_Income,Receiver_KYC_Status,Receiver_Aadhaar,Receiver_PAN,Receiver_Device_ID,Receiver_IMEI,Receiver_SIM_Number
0,254,PAYMENT,3853.04,0.0,0.00,0.00,0.00,0,CUST0113495,Union Bank of India,...,1963-03-21,2022-02-14,Teacher,1021421,Verified,816635336286,PKBQG9404Z,DEV56845324,568888046000911,8983944259644222170
1,302,CASH_IN,278263.77,204530.0,482793.77,300561.56,22297.78,0,CUST0031344,Kotak Mahindra Bank,...,1953-09-12,2013-05-03,Government Employee,517058,Verified,483341868766,PKLMH1686P,DEV53795527,593797810114220,8934614614389157455
2,39,CASH_OUT,328073.84,15865.0,0.00,2361146.10,2689219.94,0,CUST0015822,IndusInd Bank,...,1956-04-14,2022-08-25,Business,1003837,Verified,334394509598,JRJIT2176Y,DEV51994243,187252823146913,8981764420326083390
3,253,TRANSFER,85912.64,0.0,0.00,1555431.93,1641344.57,0,CUST0161046,Punjab National Bank,...,1964-12-28,2020-12-21,Shop Owner,3287827,Verified,814005872718,QYOAV2042N,DEV76704016,532751016155847,8945607117900515515
4,185,PAYMENT,17991.22,0.0,0.00,0.00,0.00,0,CUST0023815,Punjab National Bank,...,1968-09-09,2013-06-27,Doctor,1252546,Verified,314920969212,VMILZ1009D,DEV50999971,571630952634620,8912066880424968199


In [68]:
bank_transactions["Transaction_ID"] = [
    f"TXN{str(i).zfill(10)}"
    for i in range(1, len(bank_transactions) + 1)
]

In [69]:
from datetime import datetime, timedelta

In [70]:
import random

start = datetime(2025, 1, 1)
end = datetime(2025, 12, 31, 23, 59, 59)

timestamps = []

for _ in range(len(bank_transactions)):

    seconds = random.randint(
        0,
        int((end - start).total_seconds())
    )

    timestamps.append(
        start + timedelta(seconds=seconds)
    )

bank_transactions["Timestamp"] = timestamps

In [71]:
bank_transactions = bank_transactions.sort_values(
    "Timestamp"
).reset_index(drop=True)

In [72]:
mode_map = {
    "PAYMENT": "UPI",
    "TRANSFER": "IMPS",
    "CASH_OUT": "ATM Withdrawal",
    "CASH_IN": "Cash Deposit",
    "DEBIT": "POS",
}

bank_transactions["Transaction_Mode"] = (
    bank_transactions["type"]
    .map(mode_map)
    .fillna("NEFT")
)

In [73]:
bank_transactions["Transaction_Status"] = np.random.choice(
    ["SUCCESS", "FAILED", "PENDING"],
    size=len(bank_transactions),
    p=[0.97, 0.02, 0.01]
)

In [74]:
bank_transactions["Reference_Number"] = [
    "UTR" + str(random.randint(100000000000, 999999999999))
    for _ in range(len(bank_transactions))
]

In [75]:
bank_transactions[[
    "Transaction_ID",
    "Timestamp",
    "Transaction_Mode",
    "Transaction_Status",
    "Reference_Number",
    "amount"
]].head()

,Transaction_ID,Timestamp,Transaction_Mode,Transaction_Status,Reference_Number,amount
0,TXN0000004306,2025-01-01 00:13:30,Cash Deposit,SUCCESS,UTR800662653430,262014.34
1,TXN0000027442,2025-01-01 00:19:08,UPI,SUCCESS,UTR227703190014,13464.49
2,TXN0000041957,2025-01-01 00:22:04,UPI,SUCCESS,UTR284025963296,6426.26
3,TXN0000017633,2025-01-01 00:32:04,Cash Deposit,SUCCESS,UTR255061396090,40435.27
4,TXN0000081101,2025-01-01 00:38:25,ATM Withdrawal,SUCCESS,UTR231296626607,558916.57


## Finalizing our dataset:

In [76]:
bank_transactions["Transaction_ID"] = [
    f"TXN{str(i).zfill(10)}"
    for i in range(1, len(bank_transactions)+1)
]

In [77]:
from datetime import datetime, timedelta
import random

In [78]:
start_date = datetime(2025,1,1)

end_date = datetime(2025,12,31,23,59,59)

In [79]:
timestamps=[]

for i in range(len(bank_transactions)):

    seconds=random.randint(
        0,
        int((end_date-start_date).total_seconds())
    )

    timestamps.append(
        start_date+timedelta(seconds=seconds)
    )

bank_transactions["Timestamp"]=timestamps

In [80]:
bank_transactions=bank_transactions.sort_values(
    "Timestamp"
).reset_index(drop=True)

In [81]:
bank_transactions["Transaction_ID"]=[
    f"TXN{str(i).zfill(10)}"
    for i in range(1,len(bank_transactions)+1)
]

In [82]:
mode_map={

"PAYMENT":"UPI",

"TRANSFER":"IMPS",

"CASH_OUT":"ATM",

"CASH_IN":"Cash Deposit",

"DEBIT":"POS"

}

In [83]:
bank_transactions["Transaction_Mode"]=(
    bank_transactions["type"]
    .map(mode_map)
    .fillna("NEFT")
)

In [84]:
bank_transactions["Transaction_Status"]=np.random.choice(

["SUCCESS","FAILED","PENDING"],

len(bank_transactions),

p=[0.97,0.02,0.01]

)

In [85]:
bank_transactions["Currency"]="INR"

In [86]:
bank_transactions["UTR_Number"]=[

"UTR"+str(random.randint(
100000000000,
999999999999))

for i in range(len(bank_transactions))

]

In [87]:
merchant_names=[

"Amazon",

"Flipkart",

"Myntra",

"Swiggy",

"Zomato",

"Blinkit",

"Reliance Fresh",

"DMart",

"BigBasket",

"IRCTC",

"BookMyShow",

"Uber",

"Ola",

"Indian Oil",

"HP Petrol",

"Bharat Petroleum",

"Apollo Pharmacy",

"MedPlus",

"Jio Recharge",

"Airtel Recharge",

"Electricity Board",

"Water Board",

"LIC",

"Income Tax",

"Municipal Corporation"

]

In [88]:
merchant_category={

"Amazon":"Shopping",

"Flipkart":"Shopping",

"Myntra":"Shopping",

"Swiggy":"Food",

"Zomato":"Food",

"Blinkit":"Groceries",

"Reliance Fresh":"Groceries",

"DMart":"Groceries",

"BigBasket":"Groceries",

"IRCTC":"Travel",

"BookMyShow":"Entertainment",

"Uber":"Transport",

"Ola":"Transport",

"Indian Oil":"Fuel",

"HP Petrol":"Fuel",

"Bharat Petroleum":"Fuel",

"Apollo Pharmacy":"Medical",

"MedPlus":"Medical",

"Jio Recharge":"Recharge",

"Airtel Recharge":"Recharge",

"Electricity Board":"Utilities",

"Water Board":"Utilities",

"LIC":"Insurance",

"Income Tax":"Government",

"Municipal Corporation":"Government"

}

In [89]:
merchants=[]

categories=[]

In [90]:
for mode in bank_transactions["Transaction_Mode"]:

    if mode=="UPI":

        m=random.choice(merchant_names)

    elif mode=="POS":

        m=random.choice(merchant_names)

    elif mode=="ATM":

        m="ATM Withdrawal"

    elif mode=="Cash Deposit":

        m="Cash Deposit"

    else:

        m=random.choice(merchant_names)

    merchants.append(m)

    categories.append(

        merchant_category.get(

            m,

            "Bank"

        )

    )

In [91]:
bank_transactions["Merchant_Name"]=merchants

bank_transactions["Merchant_Category"]=categories

In [92]:
bank_transactions.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,Sender_Customer_ID,Sender_Bank_Name,...,Receiver_SIM_Number,Transaction_ID,Timestamp,Transaction_Mode,Transaction_Status,Reference_Number,Currency,UTR_Number,Merchant_Name,Merchant_Category
0,637,CASH_OUT,12170.46,97084.15,84913.69,114770.19,126940.64,0,CUST0088000,IndusInd Bank,...,8930541142482852126,TXN0000000001,2025-01-01 00:01:21,ATM,SUCCESS,UTR344461950738,INR,UTR123873481368,ATM Withdrawal,Bank
1,162,CASH_IN,175774.54,5542.00,181316.54,172798.84,0.00,0,CUST0173395,Axis Bank,...,8999007851962565633,TXN0000000002,2025-01-01 00:01:39,Cash Deposit,SUCCESS,UTR630307748297,INR,UTR773663556037,Cash Deposit,Bank
2,301,CASH_IN,78682.34,5924793.94,6003476.28,233950.81,155268.47,0,CUST0044406,ICICI Bank,...,8992180879000828174,TXN0000000003,2025-01-01 00:03:42,Cash Deposit,SUCCESS,UTR982070254221,INR,UTR818079976787,Cash Deposit,Bank
3,210,TRANSFER,172184.82,0.00,0.00,1719669.27,1891854.09,0,CUST0103746,Union Bank of India,...,8937280074710967420,TXN0000000004,2025-01-01 00:12:03,IMPS,SUCCESS,UTR886808663023,INR,UTR980394273515,DMart,Groceries
4,253,TRANSFER,1838884.55,11176.00,0.00,1013524.75,2852409.30,0,CUST0105884,Union Bank of India,...,8964797583753955772,TXN0000000005,2025-01-01 00:17:55,IMPS,SUCCESS,UTR109933268220,INR,UTR572341019428,Bharat Petroleum,Fuel


In [93]:
bank_transactions.drop(
    columns=[
        "step",
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest",
        "type"
    ],
    inplace=True,
    errors="ignore"
)

In [94]:
bank_transactions = (
    bank_transactions
    .sort_values("Timestamp")
    .reset_index(drop=True)
)

In [95]:
bank_transactions["Transaction_ID"] = [
    f"TXN{str(i).zfill(10)}"
    for i in range(1, len(bank_transactions) + 1)
]

In [96]:
first_columns = [
    "Transaction_ID",
    "Timestamp",
    "UTR_Number",
    "Transaction_Mode",
    "Transaction_Status",
    "Currency",
    "amount"
]

remaining_columns = [
    c for c in bank_transactions.columns
    if c not in first_columns
]

bank_transactions = bank_transactions[
    first_columns + remaining_columns
]

In [97]:
bank_transactions.rename(
    columns={
        "amount": "Transaction_Amount"
    },
    inplace=True
)

In [98]:
print("Shape :", bank_transactions.shape)
bank_transactions.head()

Shape : (100000, 51)


,Transaction_ID,Timestamp,UTR_Number,Transaction_Mode,Transaction_Status,Currency,Transaction_Amount,isFraud,Sender_Customer_ID,Sender_Bank_Name,...,Receiver_Annual_Income,Receiver_KYC_Status,Receiver_Aadhaar,Receiver_PAN,Receiver_Device_ID,Receiver_IMEI,Receiver_SIM_Number,Reference_Number,Merchant_Name,Merchant_Category
0,TXN0000000001,2025-01-01 00:01:21,UTR123873481368,ATM,SUCCESS,INR,12170.46,0,CUST0088000,IndusInd Bank,...,1596319,Verified,212481665672,NHKNJ8323P,DEV44814735,222327911347456,8930541142482852126,UTR344461950738,ATM Withdrawal,Bank
1,TXN0000000002,2025-01-01 00:01:39,UTR773663556037,Cash Deposit,SUCCESS,INR,175774.54,0,CUST0173395,Axis Bank,...,907364,Verified,272336379965,NBEZX3931V,DEV96392357,969917165658087,8999007851962565633,UTR630307748297,Cash Deposit,Bank
2,TXN0000000003,2025-01-01 00:03:42,UTR818079976787,Cash Deposit,SUCCESS,INR,78682.34,0,CUST0044406,ICICI Bank,...,1362285,Verified,329188580711,UGRDD7623J,DEV48990617,121022249994773,8992180879000828174,UTR982070254221,Cash Deposit,Bank
3,TXN0000000004,2025-01-01 00:12:03,UTR980394273515,IMPS,SUCCESS,INR,172184.82,0,CUST0103746,Union Bank of India,...,3781561,Verified,278341675629,PCIKS6953U,DEV49560694,897359907056358,8937280074710967420,UTR886808663023,DMart,Groceries
4,TXN0000000005,2025-01-01 00:17:55,UTR572341019428,IMPS,SUCCESS,INR,1838884.55,0,CUST0105884,Union Bank of India,...,2209671,Verified,575968857380,HTIIJ1627W,DEV25315074,323603801377848,8964797583753955772,UTR109933268220,Bharat Petroleum,Fuel


In [99]:
bank_transactions.to_csv(
    "bank_transactions.csv",
    index=False
)

print("bank_transactions.csv saved successfully.")

bank_transactions.csv saved successfully.
